In [1]:
!pip install librosa numpy pandas scikit-learn tensorflow


In [21]:
import os
import numpy as np
import librosa
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout


In [22]:
emotion_dict = {
    "01": "neutral",
    "02": "calm",
    "03": "happy",
    "04": "sad",
    "05": "angry",
    "06": "fear",
    "07": "disgust",
    "08": "surprise"
}


In [26]:
import zipfile

with zipfile.ZipFile("archive.zip", 'r') as zip_ref:
    zip_ref.extractall("dataset")

print("Extraction completed ✅")


Extraction completed ✅


In [27]:
import os

print(os.listdir("dataset")[:5])


['Actor_01', 'Actor_02', 'Actor_03', 'Actor_04', 'Actor_05']


In [28]:
file_paths = []
emotions = []

for actor in os.listdir(dataset_path):
    actor_path = os.path.join(dataset_path, actor)
    
    for file in os.listdir(actor_path):
        if file.endswith(".wav"):
            emotion_code = file.split("-")[2]
            emotion = emotion_dict[emotion_code]
            
            file_paths.append(os.path.join(actor_path, file))
            emotions.append(emotion)

print("Total files:", len(file_paths))


Total files: 1440


In [29]:
def extract_features(file_path):
    audio, sample_rate = librosa.load(file_path, duration=3, offset=0.5)
    
    mfcc = librosa.feature.mfcc(y=audio, sr=sample_rate, n_mfcc=40)
    chroma = librosa.feature.chroma_stft(y=audio, sr=sample_rate)
    mel = librosa.feature.melspectrogram(y=audio, sr=sample_rate)
    
    mfcc = np.mean(mfcc.T, axis=0)
    chroma = np.mean(chroma.T, axis=0)
    mel = np.mean(mel.T, axis=0)
    
    return np.hstack((mfcc, chroma, mel))


In [35]:
X = []
y = []

for file_path, emotion in zip(file_paths, emotions):
    features = extract_features(file_path)
    X.append(features)
    y.append(emotion)

X = np.array(X)
y = np.array(y)

print("Feature shape:", X.shape)


Feature shape: (1440, 180)


In [36]:
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)
y_categorical = to_categorical(y_encoded)


In [37]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y_categorical, test_size=0.2, random_state=42
)


In [38]:
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


In [39]:
model = Sequential()

model.add(Dense(512, input_shape=(X_train.shape[1],), activation='relu'))
model.add(Dropout(0.4))

model.add(Dense(256, activation='relu'))
model.add(Dropout(0.4))

model.add(Dense(128, activation='relu'))
model.add(Dropout(0.3))

model.add(Dense(8, activation='softmax'))


C:\Users\KANAL PATEL\Anaconda3_New\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [40]:
model.compile(
    loss='categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

In [41]:
history = model.fit(
    X_train,
    y_train,
    epochs=70,
    batch_size=16,
    validation_data=(X_test, y_test)
)

Epoch 1/70
72/72 ━━━━━━━━━━━━━━━━━━━━ 6s 20ms/step - accuracy: 0.2135 - loss: 2.0254 - val_accuracy: 0.4062 - val_loss: 1.6308
Epoch 2/70
72/72 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.3264 - loss: 1.7724 - val_accuracy: 0.3993 - val_loss: 1.5536
Epoch 3/70
72/72 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.3845 - loss: 1.6204 - val_accuracy: 0.4653 - val_loss: 1.4578
Epoch 4/70
72/72 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.4149 - loss: 1.5467 - val_accuracy: 0.4653 - val_loss: 1.4334
Epoch 5/70
72/72 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.4566 - loss: 1.4446 - val_accuracy: 0.4965 - val_loss: 1.3465
Epoch 6/70
72/72 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.5174 - loss: 1.3382 - val_accuracy: 0.5278 - val_loss: 1.3208
Epoch 7/70
72/72 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step - accuracy: 0.5425 - loss: 1.2800 - val_accuracy: 0.5035 - val_loss: 1.2317
Epoch 8/70
72/72 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.5807 - loss: 1.1936 - val_accuracy: 0.5451 - v

In [42]:
loss, accuracy = model.evaluate(X_test, y_test)
print("Final Test Accuracy:", accuracy)

9/9 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - accuracy: 0.6528 - loss: 1.5710 
Final Test Accuracy: 0.6527777910232544
